# PEN OIS Fair Value Model
**BCRP meeting-path pricer · ACT/360 TNA · Zero-coupon OIS**

Prices PEN overnight index swaps given user-defined BCRP rate paths, computes fair-value (FV) rates for 3M/6M/9M/12M tenors, bootstraps a piecewise-constant forward curve, extracts market-implied path probabilities via constrained LS and maximum-entropy, and shows everything in one interactive widget.

> **Day count:** ACT/360 · **Convention:** TNA (nominal annual) · **Float reference:** TIBO · **Settlement:** T+2 ModFol

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
from datetime import date, timedelta
import calendar as _cal
import copy as _copy

warnings.filterwarnings("ignore")

# ── USER CONFIG ──────────────────────────────────────────────
# xlsx with Sheet1 columns: dates, 3mo, 6mo, 9mo, 12mo  (rates in TEA %)
DATA_PATH = "ois_data.xlsx"

TIBO_DEF = 4.25   # current TIBO overnight rate (% TNA)
MKT_DEF  = {"3M": 4.15, "6M": 4.10, "9M": 4.095, "12M": 4.095}

TENORS  = ["3M", "6M", "9M", "12M"]
N_MTG   = 14     # forward meetings shown in dashboard

# 2026 confirmed BCRP meeting dates
BCRP_2026 = [
    date(2026,  1,  8), date(2026,  2, 12), date(2026,  3, 12),
    date(2026,  4,  9), date(2026,  5, 14), date(2026,  6, 11),
    date(2026,  7,  9), date(2026,  8, 13), date(2026,  9, 10),
    date(2026, 10,  7), date(2026, 11, 12), date(2026, 12, 10),
]


In [ ]:
# ── Gauss Easter algorithm ──────────────────────────────────
def _easter(y):
    a = y % 19; b = y // 100; c = y % 100
    d = b // 4; e = b % 4; f = (b + 8) // 25; g = (b - f + 1) // 3
    h = (19*a + b - d - g + 15) % 30
    i = c // 4; k = c % 4
    l = (32 + 2*e + 2*i - h - k) % 7
    m = (a + 11*h + 22*l) // 451
    mo = (h + l - 7*m + 114) // 31
    dy = (h + l - 7*m + 114) % 31 + 1
    return date(y, mo, dy)

def _pe_hols(y):
    ea = _easter(y)
    return {
        date(y, 1, 1), date(y, 5, 1), date(y, 6, 29),
        date(y, 7, 28), date(y, 7, 29), date(y, 8, 30),
        date(y, 10, 8), date(y, 11, 1), date(y, 12, 8), date(y, 12, 25),
        ea - timedelta(3), ea - timedelta(2),
    }

def _nth(y, mo, wd, n):
    d = date(y, mo, 1); cnt = 0
    while True:
        if d.weekday() == wd:
            cnt += 1
            if cnt == n: return d
        d += timedelta(1)

def _last(y, mo, wd):
    d = (date(y, mo+1, 1) if mo < 12 else date(y+1, 1, 1)) - timedelta(1)
    while d.weekday() != wd: d -= timedelta(1)
    return d

def _us_hols(y):
    return {
        date(y, 1, 1), date(y, 7, 4), date(y, 11, 11), date(y, 12, 25),
        _nth(y, 1, 0, 3), _nth(y, 2, 0, 3), _last(y, 5, 0),
        _nth(y, 9, 0, 1), _nth(y, 10, 0, 2), _nth(y, 11, 3, 4),
    }

_HC = {}
def _hols(y):
    if y not in _HC:
        _HC[y] = _pe_hols(y) | _us_hols(y)
    return _HC[y]

def is_bday(d):
    return d.weekday() < 5 and d not in _hols(d.year)

def mod_fol(d):
    orig = d.month; t = d
    while not is_bday(t): t += timedelta(1)
    if t.month != orig:
        t = d
        while not is_bday(t): t -= timedelta(1)
    return t

def add_months(d, n):
    m = d.month - 1 + n
    y = d.year + m // 12
    m = m % 12 + 1
    return date(y, m, min(d.day, _cal.monthrange(y, m)[1]))

def spot(val_date):
    t = val_date + timedelta(1)
    while not is_bday(t): t += timedelta(1)
    t += timedelta(1)
    while not is_bday(t): t += timedelta(1)
    return t

def tenor_end(start, months):
    return mod_fol(add_months(start, months))

print("Calendar utils ready.")


In [ ]:
def _second_thu(y, mo):
    d = date(y, mo, 1); cnt = 0
    while True:
        if d.weekday() == 3:
            cnt += 1
            if cnt == 2: return d
        d += timedelta(1)

def get_meetings(val_date, n=N_MTG):
    pool = sorted(
        list(BCRP_2026)
        + [_second_thu(y, mo) for y in range(2027, 2036) for mo in range(1, 13)]
    )
    return [m for m in pool if m > val_date][:n]

def price_ois(tibo, meetings, bp_ch, start_d, end_d):
    # ACT/360, zero-coupon, TNA quote convention
    # Float = prod_j (1 + r_j/360)^m_j  where m_j = calendar days at rate r_j
    # K* = (Float - 1) * 360 / D
    D = (end_d - start_d).days
    if D <= 0:
        return tibo

    # Cumulative rate schedule: (effective_date, rate_value)
    cum = tibo
    eff_sched = []
    for mtg, bps in zip(meetings, bp_ch):
        cum = round(cum + bps / 100.0, 8)
        eff_sched.append((mtg + timedelta(1), cum))

    def rate_on(d):
        r = tibo
        for eff, rv in eff_sched:
            if eff <= d: r = rv
            else: break
        return r

    bkpts = sorted(
        {start_d, end_d}
        | {eff for eff, _ in eff_sched if start_d < eff < end_d}
    )

    ff = 1.0
    for i in range(len(bkpts) - 1):
        r  = rate_on(bkpts[i]) / 100.0
        dy = (bkpts[i+1] - bkpts[i]).days
        ff *= (1.0 + r / 360.0) ** dy

    return round((ff - 1.0) * 360.0 / D * 100.0, 6)

def price_all(scenarios, tibo, val_date):
    st   = spot(val_date)
    mtgs = get_meetings(val_date)
    ends = {t: tenor_end(st, m) for t, m in [("3M",3),("6M",6),("9M",9),("12M",12)]}
    fv   = {}
    for sc in scenarios:
        bps = (sc["moves"] + [0] * N_MTG)[:len(mtgs)]
        fv[sc["name"]] = {t: price_ois(tibo, mtgs, bps, st, ends[t]) for t in TENORS}
    return fv, mtgs, st, ends

print("Pricing engine ready.")


In [ ]:
def bootstrap_forwards(tibo, mkt, val_date):
    st    = spot(val_date)
    nodes = [("3M",3), ("6M",6), ("9M",9), ("12M",12)]
    ends  = [(t, tenor_end(st, m)) for t, m in nodes]
    gf    = {t: 1.0 + mkt[t]/100.0 * (end_d - st).days / 360.0 for t, end_d in ends}
    fwds  = {}
    prev_end = st; prev_gf = 1.0; prev_lbl = "0M"
    for t, end_d in ends:
        days = (end_d - prev_end).days
        seg  = gf[t] / prev_gf
        fwds[f"{prev_lbl}–{t}"] = round((seg - 1.0) * 360.0 / days * 100.0, 6)
        prev_end = end_d; prev_gf = gf[t]; prev_lbl = t
    return fwds

def build_fv_matrix(fv_dict):
    names = list(fv_dict.keys())
    P = np.array([[fv_dict[n][t] for n in names] for t in TENORS])
    return P, names

def solve_ls(P, m_vec):
    n  = P.shape[1]
    w0 = np.ones(n) / n
    res = minimize(
        lambda w: float(np.sum((P @ w - m_vec)**2)),
        w0,
        jac=lambda w: 2.0 * P.T @ (P @ w - m_vec),
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n,
        constraints=[{"type": "eq", "fun": lambda w: float(w.sum() - 1.0)}],
        options={"ftol": 1e-12, "maxiter": 2000},
    )
    return res.x if res.success else w0

def solve_maxent(P, m_vec, eps=0.01):
    n  = P.shape[1]
    w0 = np.ones(n) / n
    def _neg_ent(w):
        ws = np.clip(w, 1e-12, 1.0)
        return float(np.sum(ws * np.log(ws)))
    def _neg_ent_jac(w):
        return np.log(np.clip(w, 1e-12, 1.0)) + 1.0
    res = minimize(
        _neg_ent, w0, jac=_neg_ent_jac,
        method="SLSQP",
        bounds=[(1e-9, 1.0)] * n,
        constraints=[
            {"type": "eq",   "fun": lambda w: float(w.sum() - 1.0)},
            {"type": "ineq", "fun": lambda w: float(eps - np.max(np.abs(P @ w - m_vec)))},
        ],
        options={"ftol": 1e-12, "maxiter": 3000},
    )
    return res.x if res.success else solve_ls(P, m_vec)

def mtg_probs(weights, scenarios, meetings, names):
    sc_map = {s["name"]: s for s in scenarios}
    rows   = []; cum = 0.0
    for i, mtg in enumerate(meetings):
        pc = ph = pk = em = 0.0
        for name, w in zip(names, weights):
            mv = sc_map[name]["moves"][i] if i < len(sc_map[name]["moves"]) else 0
            if mv < 0:   pc += w
            elif mv > 0: pk += w
            else:        ph += w
            em += w * mv
        cum += em
        rows.append({
            "Meeting":      mtg.strftime("%d %b %y"),
            "P(Cut)%":      round(pc * 100, 1),
            "P(Hold)%":     round(ph * 100, 1),
            "P(Hike)%":     round(pk * 100, 1),
            "E[Move]":      round(em, 2),
            "Cum E[Move]":  round(cum, 2),
        })
    return rows

print("Solvers ready.")


In [ ]:
def _sc(name, idxs, bps=-25, n=N_MTG):
    mv = [0] * n
    for i in idxs:
        if i < n: mv[i] = bps
    return {"name": name, "weight": 0.0, "moves": mv}

BASE_SCENARIOS = [
    _sc("Hold",              []),
    _sc("Apr -25",           [0]),
    _sc("May -25",           [1]),
    _sc("Jun -25",           [2]),
    _sc("Jul -25",           [3]),
    _sc("Apr/Jul -25",       [0, 3]),
    _sc("Apr/Jun -25",       [0, 2]),
    _sc("Jun/Sep -25",       [2, 5]),
    _sc("Apr/Jul/Oct -25",   [0, 3, 6]),
    _sc("Apr -50",           [0], bps=-50),
    _sc("Apr/May/Jun -25",   [0, 1, 2]),
    _sc("Apr +25",           [0], bps=+25),
    _sc("Apr/Jun/Sep -25",   [0, 2, 5]),
    _sc("May/Sep -25",       [1, 5]),
]

# Equal starting weights
_eq = round(100.0 / len(BASE_SCENARIOS), 2)
for _s in BASE_SCENARIOS:
    _s["weight"] = _eq
BASE_SCENARIOS[-1]["weight"] = round(100.0 - _eq * (len(BASE_SCENARIOS) - 1), 2)

print(f"Loaded {len(BASE_SCENARIOS)} base scenarios.")


In [ ]:
_CSS = """
<style>
.oit {font-family:'Segoe UI',Arial,sans-serif;border-collapse:collapse;
      font-size:11.5px;width:100%;}
.oit th {background:#1f2d40;color:#fff;padding:5px 9px;text-align:center;
         white-space:nowrap;font-weight:600;letter-spacing:.3px;}
.oit td {padding:3px 8px;text-align:center;border:1px solid #e2e8f0;white-space:nowrap;}
.oit tr:nth-child(even) td {background:#f8fafc;}
.cn   {text-align:left!important;padding-left:10px!important;font-weight:500;}
.rsep td {border-top:2px solid #94a3b8!important;}
.rmkt td {background:#dcfce7!important;font-weight:700;}
.rfv  td {background:#dbeafe!important;font-weight:700;}
.rdif td {font-weight:700;}
.rst  td {color:#64748b;font-style:italic;}
</style>
"""

def _bg(v):
    if v <= -2.0: return "background:#fecaca;"
    if v >=  2.0: return "background:#bbf7d0;"
    return ""

def render_fv_table(fv_dict, mkt, user_fv, scenarios):
    names  = list(fv_dict.keys())
    sc_map = {s["name"]: s for s in scenarios}
    hdr    = "".join(f"<th>{t}</th>" for t in TENORS)
    html   = _CSS + (
        '<table class="oit"><thead>'
        f'<tr><th style="text-align:left;min-width:130px">Path</th>'
        f'<th style="min-width:48px">Wt%</th>{hdr}</tr>'
        "</thead><tbody>\n"
    )
    for name in names:
        wt    = sc_map.get(name, {}).get("weight", 0.0)
        cells = ""
        prev  = None
        for t in TENORS:
            v  = fv_dict[name][t]
            dm = (v - mkt[t]) * 100
            cells += f'<td style="{_bg(dm)}">{v:.4f}</td>'
        html += f'<tr><td class="cn">{name}</td><td>{wt:.1f}</td>{cells}</tr>\n'

    mkt_c = "".join(f"<td>{mkt[t]:.4f}</td>" for t in TENORS)
    html += f'<tr class="rsep rmkt"><td class="cn">Mkt</td><td>100</td>{mkt_c}</tr>\n'

    fvc = "".join(f"<td>{user_fv[t]:.4f}</td>" for t in TENORS)
    html += f'<tr class="rfv"><td class="cn">User FV (wtd)</td><td>—</td>{fvc}</tr>\n'

    ev_c = ""
    for t in TENORS:
        v = round((mkt[t] - user_fv[t]) * 100, 2)
        ev_c += f'<td style="{_bg(v)};font-weight:700">{v:+.2f}</td>'
    html += f'<tr class="rdif rsep"><td class="cn">Weighted EV (bps)</td><td>—</td>{ev_c}</tr>\n'

    hi_c = lo_c = ""
    for t in TENORS:
        vs   = [(fv_dict[n][t] - mkt[t]) * 100 for n in names]
        hi_v = round(max(vs), 2); lo_v = round(min(vs), 2)
        hi_c += f'<td style="{_bg(hi_v)}">{hi_v:+.2f}</td>'
        lo_c += f'<td style="{_bg(lo_v)}">{lo_v:+.2f}</td>'
    html += f'<tr class="rst"><td class="cn">High Delta Active (bps)</td><td>—</td>{hi_c}</tr>\n'
    html += f'<tr class="rst"><td class="cn">Low Delta Active (bps)</td><td>—</td>{lo_c}</tr>\n'
    html += "</tbody></table>"
    return html

def render_fwd_table(fwds):
    hdr  = "".join(f"<th>{seg}</th>" for seg in fwds)
    vals = "".join(f"<td>{v:.4f}%</td>" for v in fwds.values())
    return _CSS + (
        '<table class="oit" style="width:auto">'
        f"<thead><tr><th style='text-align:left'>Segment</th>{hdr}</tr></thead>"
        f"<tbody><tr><td class='cn'>Fwd ON Rate (% TNA)</td>{vals}</tr></tbody>"
        "</table>"
    )

def render_prob_tables(ls_rows, me_rows, names, ls_w, me_w):
    # Path weights
    pw_rows = "".join(
        f"<tr><td class='cn'>{n}</td><td>{lw*100:.1f}</td><td>{mw*100:.1f}</td></tr>"
        for n, lw, mw in sorted(zip(names, ls_w, me_w), key=lambda x: -x[1])
    )
    pw_html = _CSS + (
        '<table class="oit" style="width:auto;display:inline-table;margin-right:28px">'
        "<thead><tr><th style='text-align:left'>Path</th>"
        "<th>LS Wt%</th><th>MaxEnt Wt%</th></tr></thead>"
        f"<tbody>{pw_rows}</tbody></table>"
    )

    # Per-meeting summary
    mtg_hdr = (
        "<th>Meeting</th>"
        "<th>LS P(Cut)</th><th>LS P(Hold)</th><th>LS P(Hike)</th>"
        "<th>LS E[mv]</th><th>LS Cum</th>"
        "<th>ME P(Cut)</th><th>ME P(Hold)</th><th>ME P(Hike)</th>"
        "<th>ME E[mv]</th><th>ME Cum</th>"
    )
    mtg_rows = ""
    for lr, mr in zip(ls_rows, me_rows):
        mtg_rows += (
            f"<tr><td class='cn'>{lr['Meeting']}</td>"
            f"<td>{lr['P(Cut)%']}</td><td>{lr['P(Hold)%']}</td><td>{lr['P(Hike)%']}</td>"
            f"<td>{lr['E[Move]']:+.2f}</td><td>{lr['Cum E[Move]']:+.2f}</td>"
            f"<td>{mr['P(Cut)%']}</td><td>{mr['P(Hold)%']}</td><td>{mr['P(Hike)%']}</td>"
            f"<td>{mr['E[Move]']:+.2f}</td><td>{mr['Cum E[Move]']:+.2f}</td>"
            "</tr>\n"
        )
    mtg_html = _CSS + (
        f'<table class="oit"><thead><tr>{mtg_hdr}</tr></thead>'
        f"<tbody>{mtg_rows}</tbody></table>"
    )
    return pw_html, mtg_html

print("Table renderers ready.")


In [ ]:
def _tea_to_tna(tea_pct):
    # TEA to TNA: r_TNA = ((1 + r_TEA)^(1/360) - 1) * 360
    return ((1.0 + tea_pct / 100.0) ** (1.0 / 360.0) - 1.0) * 360.0 * 100.0

def _load_hist(data_path, val_date):
    try:
        xl = pd.read_excel(data_path, sheet_name="Sheet1")
        xl.columns = [c.strip().lower() for c in xl.columns]
        xl["dates"] = pd.to_datetime(xl["dates"])
        xl = xl.sort_values("dates")
        for col in ["3mo", "6mo", "9mo", "12mo"]:
            if col in xl.columns:
                xl[col] = xl[col].apply(_tea_to_tna)
        xl = xl.rename(columns={"3mo": "3M", "6mo": "6M", "9mo": "9M", "12mo": "12M"})
        cutoff = pd.Timestamp(val_date) - pd.DateOffset(years=1)
        return xl[xl["dates"] >= cutoff][["dates", "3M", "6M", "9M", "12M"]]
    except Exception:
        return _synth_hist(val_date)

def _synth_hist(val_date):
    end   = pd.Timestamp(val_date)
    start = end - pd.DateOffset(years=1)
    dates = pd.bdate_range(start, end)
    n     = len(dates)
    rng   = np.random.default_rng(99)
    rows  = {"dates": dates}
    offsets = {"3M": 0.50, "6M": 0.55, "9M": 0.56, "12M": 0.56}
    for t, off in offsets.items():
        walk = np.cumsum(rng.normal(0, 0.025, n))
        walk = walk - walk[-1]
        rows[t] = np.clip(MKT_DEF[t] + off + walk * 0.6, 1.5, 8.5)
    return pd.DataFrame(rows)

_LINE_COLS = ["#7c3aed", "#059669", "#2563eb", "#db2777", "#d97706"]

def build_chart(hist_df, fv_dict, mkt, user_fv, ls_w, names, tenor):
    fig = go.Figure()

    # Historical realized line
    fig.add_trace(go.Scatter(
        x=hist_df["dates"], y=hist_df[tenor],
        mode="lines", name=f"PEN OIS {tenor}",
        line=dict(color="#1e3a5f", width=2.2),
        hovertemplate="%{x|%d %b %y}: %{y:.3f}%<extra></extra>",
    ))

    # Market level — dashed dark
    fig.add_hline(
        y=mkt[tenor], line_color="#374151", line_dash="dash", line_width=1.5,
        annotation_text=f"Mkt: {mkt[tenor]:.3f}%",
        annotation_position="top right",
        annotation_font=dict(color="#374151", size=10),
    )

    # User FV — solid orange
    fig.add_hline(
        y=user_fv[tenor], line_color="#ea580c", line_dash="solid", line_width=2.0,
        annotation_text=f"FV: {user_fv[tenor]:.3f}%",
        annotation_position="bottom right",
        annotation_font=dict(color="#ea580c", size=10),
    )

    # Top-5 path FV levels (by LS weight), deduped
    top5_idx = np.argsort(ls_w)[-5:][::-1]
    seen     = {round(mkt[tenor], 4), round(user_fv[tenor], 4)}
    k = 0
    for idx in top5_idx:
        if ls_w[idx] < 1e-4: continue
        lv = round(fv_dict[names[idx]][tenor], 4)
        if lv in seen: continue
        seen.add(lv)
        col = _LINE_COLS[k % len(_LINE_COLS)]
        fig.add_hline(
            y=lv, line_color=col, line_dash="dot", line_width=1.3,
            annotation_text=f"{names[idx]}: {lv:.3f}%",
            annotation_position="top right" if k % 2 == 0 else "bottom right",
            annotation_font=dict(color=col, size=9),
        )
        k += 1

    fig.update_layout(
        title=dict(
            text=f"PEN OIS {tenor} — Market vs Path FV Levels  |  1Y Lookback",
            font=dict(size=14, color="#1e3a5f"), x=0.0,
        ),
        paper_bgcolor="white", plot_bgcolor="white",
        xaxis=dict(
            title="", gridcolor="#f1f5f9", linecolor="#cbd5e1",
            showgrid=True, zeroline=False,
        ),
        yaxis=dict(
            title="Rate (% TNA)", gridcolor="#f1f5f9", linecolor="#cbd5e1",
            tickformat=".3f", showgrid=True, zeroline=False,
        ),
        font=dict(family="Arial", size=11, color="#334155"),
        legend=dict(orientation="h", y=1.02, x=0, font=dict(size=9)),
        height=460, margin=dict(l=60, r=190, t=65, b=50),
        hovermode="x unified",
    )
    return fig

print("Chart builder ready.")


In [ ]:
# ── Market data input panel ──────────────────────────────────
_fw = "160px"; _ds = {"description_width": "75px"}
w_tibo  = widgets.FloatText(value=TIBO_DEF, description="TIBO ON:", step=0.01,
                             layout=widgets.Layout(width=_fw), style=_ds)
w_3m    = widgets.FloatText(value=MKT_DEF["3M"],  description="3M OIS:",  step=0.001,
                             layout=widgets.Layout(width=_fw), style=_ds)
w_6m    = widgets.FloatText(value=MKT_DEF["6M"],  description="6M OIS:",  step=0.001,
                             layout=widgets.Layout(width=_fw), style=_ds)
w_9m    = widgets.FloatText(value=MKT_DEF["9M"],  description="9M OIS:",  step=0.001,
                             layout=widgets.Layout(width=_fw), style=_ds)
w_12m   = widgets.FloatText(value=MKT_DEF["12M"], description="12M OIS:", step=0.001,
                             layout=widgets.Layout(width=_fw), style=_ds)
w_vdate = widgets.DatePicker(value=date.today(), description="Val Date:",
                              layout=widgets.Layout(width="220px"),
                              style={"description_width": "75px"})

# ── Path editor grid ─────────────────────────────────────────
_val0    = date.today()
_mtgs0   = get_meetings(_val0)
_mlabels = [m.strftime("%b%y") for m in _mtgs0]

def _make_row_ws(sc):
    n_w = widgets.Text(value=sc["name"], layout=widgets.Layout(width="118px"))
    wt  = widgets.FloatText(value=sc["weight"], step=0.1,
                             layout=widgets.Layout(width="56px"))
    mv  = [widgets.IntText(
                value=int(sc["moves"][i]) if i < len(sc["moves"]) else 0,
                layout=widgets.Layout(width="50px"))
           for i in range(N_MTG)]
    return {"n": n_w, "w": wt, "m": mv}

def _hdr_row():
    s = "font-weight:700;font-size:11px;font-family:Arial;text-align:center;"
    items = (
        [widgets.HTML(f'<div style="{s}width:118px;text-align:left">Path</div>')]
        + [widgets.HTML(f'<div style="{s}width:56px">Wt%</div>')]
        + [widgets.HTML(f'<div style="{s}width:50px">{lbl}</div>') for lbl in _mlabels]
    )
    return widgets.HBox(items)

def _row_hbox(rw):
    return widgets.HBox([rw["n"], rw["w"]] + rw["m"])

def _read_sc(rw_list):
    return [{"name": r["n"].value, "weight": float(r["w"].value),
             "moves": [int(w.value) for w in r["m"]]} for r in rw_list]

row_ws   = [_make_row_ws(_copy.deepcopy(sc)) for sc in BASE_SCENARIOS]
hdr_box  = _hdr_row()
grid_vbx = widgets.VBox([_row_hbox(r) for r in row_ws])

# ── Buttons ──────────────────────────────────────────────────
btn_upd   = widgets.Button(description="Update Output",  button_style="info",
                            layout=widgets.Layout(width="142px", height="32px"))
btn_prices= widgets.Button(description="Show Prices",    button_style="",
                            layout=widgets.Layout(width="118px", height="32px"))
btn_add   = widgets.Button(description="+ Add Row",      button_style="success",
                            layout=widgets.Layout(width="100px", height="32px"))
w_tenor   = widgets.Dropdown(options=TENORS, value="3M", description="Chart tenor:",
                              style={"description_width": "90px"},
                              layout=widgets.Layout(width="200px"))

out_info  = widgets.Output()
out_fv    = widgets.Output()
out_fwd   = widgets.Output()
out_prob  = widgets.Output()
out_chart = widgets.Output()

# ── Compute & render ──────────────────────────────────────────
def run(_=None):
    scenarios = _read_sc(row_ws)
    tibo  = float(w_tibo.value)
    mkt   = {"3M": float(w_3m.value), "6M": float(w_6m.value),
             "9M": float(w_9m.value), "12M": float(w_12m.value)}
    vdate = w_vdate.value or date.today()

    fv_dict, mtgs, st, ends = price_all(scenarios, tibo, vdate)

    total_w = max(sum(s["weight"] for s in scenarios) / 100.0, 1e-9)
    user_fv = {
        t: round(sum(s["weight"] / 100.0 * fv_dict[s["name"]][t]
                     for s in scenarios) / total_w, 4)
        for t in TENORS
    }

    fwds   = bootstrap_forwards(tibo, mkt, vdate)
    P, nms = build_fv_matrix(fv_dict)
    m_vec  = np.array([mkt[t] for t in TENORS])
    ls_w   = solve_ls(P, m_vec)
    me_w   = solve_maxent(P, m_vec)
    ls_rows = mtg_probs(ls_w, scenarios, mtgs, nms)
    me_rows = mtg_probs(me_w, scenarios, mtgs, nms)
    pw_html, mp_html = render_prob_tables(ls_rows, me_rows, nms, ls_w, me_w)
    hist_df = _load_hist(DATA_PATH, vdate)

    with out_info:
        clear_output(wait=True)
        twt = sum(s["weight"] for s in scenarios)
        display(HTML(
            f'<p style="font-family:Arial;font-size:11px;color:#64748b;margin:2px 0">'
            f'Updated PEN OIS with <b>{len(scenarios)}</b> paths and '
            f'<b>{len(TENORS)}</b> tenors. Anchor: TIBO {tibo:.2f}% TNA. '
            f'Val date: {vdate}. Total weight: <b>{twt:.1f}%</b>. '
            f'Spot start: {st}.</p>'
        ))

    with out_fv:
        clear_output(wait=True)
        display(HTML(
            "<h4 style='font-family:Arial;color:#1e3a5f;margin:12px 0 4px'>"
            "Path FV Table</h4>"
        ))
        display(HTML(render_fv_table(fv_dict, mkt, user_fv, scenarios)))

    with out_fwd:
        clear_output(wait=True)
        display(HTML(
            "<h4 style='font-family:Arial;color:#1e3a5f;margin:12px 0 4px'>"
            "Piecewise Constant Forward Bootstrap</h4>"
        ))
        display(HTML(render_fwd_table(fwds)))

    with out_prob:
        clear_output(wait=True)
        display(HTML(
            "<h4 style='font-family:Arial;color:#1e3a5f;margin:12px 0 4px'>"
            "Market-Implied Path Weights</h4>"
        ))
        display(HTML(pw_html))
        display(HTML(
            "<h4 style='font-family:Arial;color:#1e3a5f;margin:12px 0 4px'>"
            "Per-Meeting Probability Summary  (LS | MaxEnt)</h4>"
        ))
        display(HTML(mp_html))

    with out_chart:
        clear_output(wait=True)
        build_chart(hist_df, fv_dict, mkt, user_fv, ls_w, nms, w_tenor.value).show()

def add_row(_=None):
    rw = _make_row_ws({"name": "New Path", "weight": 0.0, "moves": [0]*N_MTG})
    row_ws.append(rw)
    grid_vbx.children = tuple(_row_hbox(r) for r in row_ws)

btn_upd.on_click(run)
btn_prices.on_click(run)
btn_add.on_click(add_row)
w_tenor.observe(lambda c: run() if c["name"] == "value" else None, names="value")

# ── Layout ────────────────────────────────────────────────────
_title = widgets.HTML(
    "<h2 style='font-family:\"Segoe UI\",Arial;color:#1e3a5f;margin:0 0 3px'>"
    "PEN OIS Fair Value Model</h2>"
    "<p style='font-family:Arial;font-size:11px;color:#94a3b8;margin:0 0 8px'>"
    "BCRP meeting-path pricer · ACT/360 TNA · Zero-coupon · Float: TIBO</p>"
)
_mkt_box = widgets.VBox([
    widgets.HTML("<b style='font-family:Arial;font-size:12px;color:#1e3a5f'>"
                 "Market Data</b>"),
    widgets.HBox([w_tibo, w_3m, w_6m]),
    widgets.HBox([w_9m, w_12m, w_vdate]),
], layout=widgets.Layout(border="1px solid #e2e8f0", padding="8px 10px",
                          margin="0 0 8px", border_radius="6px",
                          background="#fafbfc"))
_ctrl_row = widgets.HBox(
    [btn_upd, btn_prices, btn_add, w_tenor],
    layout=widgets.Layout(margin="6px 0 0", align_items="center"),
)
_sec = lambda t: widgets.HTML(
    f"<h4 style='font-family:Arial;color:#1e3a5f;margin:14px 0 4px'>{t}</h4>"
)
_editor = widgets.VBox(
    [hdr_box, grid_vbx, _ctrl_row],
    layout=widgets.Layout(border="1px solid #e2e8f0", padding="8px 10px",
                           border_radius="6px", background="#fafbfc"),
)
_dash = widgets.VBox(
    [_title, _mkt_box, _editor, out_info,
     _sec("Path FV Table"), out_fv,
     _sec("Forward Bootstrap"), out_fwd,
     _sec("Market-Implied Probabilities"), out_prob,
     _sec("Rate Chart"), out_chart],
    layout=widgets.Layout(max_width="1500px"),
)

display(_dash)
run()
